In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [ ]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

In [ ]:
wapor_aeti = (
    ee.ImageCollection("FAO/WAPOR/3/L1_AETI_D")
    .filterDate("2018-01-01", "2024-12-31")
    .filterBounds(panama_geom)
    .select("L1-AETI-D")  # Fix 1: Correct band name
)

def calculate_yearly_aeti(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    # Multiply by 0.1 scale factor to get total mm/year
    annual_sum = wapor_aeti.filterDate(start, end).sum().multiply(0.1).float().rename(f"aeti_{year}")
    return annual_sum

yearly_aeti = ee.Image.cat([calculate_yearly_aeti(yr) for yr in aeti_years])